In [1]:
# Jaxlib
import jax
from jax import lax
from jax.experimental import checkify
from jax import random as jrnd
from jax import numpy as jnp
from jax import tree_util as jtu
from jax import tree as jt

from jax import vmap
jax.config.update('jax_enable_x64', True)

# Others
from matplotlib import pyplot as plt
from collections.abc import Callable, Iterable

# Mine
from gm_utils import *

['/src', '/opt/homebrew/Cellar/python@3.13/3.13.6/Frameworks/Python.framework/Versions/3.13/lib/python313.zip', '/opt/homebrew/Cellar/python@3.13/3.13.6/Frameworks/Python.framework/Versions/3.13/lib/python3.13', '/opt/homebrew/Cellar/python@3.13/3.13.6/Frameworks/Python.framework/Versions/3.13/lib/python3.13/lib-dynload', '', '/Users/ianmessa/Desktop/Scripts/thesis/.venv/lib/python3.13/site-packages']


ModuleNotFoundError: No module named 'numerics'

In [ ]:
import requests
from ASK14 import f_ASK14
from BSSA14 import f_BSSA14 
from CY14 import f_CY14 
from CB14 import f_CB14
from Idriss14 import f_Idriss14
from epistemic_AAY14 import f_epistemic_AAY14

def test_model(scn:Scenario, name, model):
    my_T, my_lnSA, my_sigma = model(scn)

    # API
    url_base = 'https://earthquake.usgs.gov/ws/nshmp/gmm/spectra?'
    url_inputs = (name, scn.Mw, scn.R_jb, scn.R_rup, scn.R_x, scn.dip, scn.width, scn.z_tor, scn.z_hyp, scn.rake, scn.vs30, scn.z1p0, scn.z2p5)
    url_addn = 'gmm=%s&Mw=%s&rJB=%s&rRup=%s&rX=%s&dip=%s&width=%s&z_tor=%s&z_hyp=%s&rake=%s&vs30=%s&z1p0=%s&z2p5=%s'%url_inputs
    json = requests.get(url_base + url_addn).json()
    print(json)
    api_T = json['response']['means']['data'][0]['data']['sa']['xs']
    api_PGA, api_PGV = [json['response']['means']['data'][0]['data'][key] for key in ['pga', 'pgv']]
    if api_PGV is None:
        api_PGV = jnp.nan
    api_SA = json['response']['means']['data'][0]['data']['sa']['ys']
    api_T, api_SA= [jnp.array(_) for _ in [api_T, api_SA]]
    api_T = jnp.append(api_T, jnp.array([-1, -2]))
    api_SA = jnp.append(api_SA, jnp.array([api_PGA, api_PGV]))
    api_lnSA = jnp.log(api_SA)

    plt.plot(api_T, api_lnSA, c = 'k')
    plt.plot(my_T, my_lnSA, c = 'r', ls = '--')
    plt.xscale('log')
    plt.suptitle(name)
    plt.show()



f_Idriss_plus_ep = f_epistemic_AAY14(f_Idriss14, 1.645)
f_Idriss_minus_ep = f_epistemic_AAY14(f_Idriss14, -1.645)
scn = gm_scenario(7., 85., 40., 2., 2., 2., 2., 760, 1.3, 1.7, 0.4, 0.1)

NameError: name 'gm_scenario' is not defined